# **Consolidación de datos de localización por segmento para visualización multiescalar**

Este script completa el proceso de geocodificación inversa al unir todos los archivos `.txt` generados por cada segmento en un solo archivo tabular consolidado. Esto permite **analizar y visualizar de manera integrada los resultados espaciales (vías, automóviles, anomalías)** desde distintos niveles geográficos.

---

### 🎯 Objetivo del proceso:

Unificar los resultados de localización obtenidos por segmento para permitir su análisis **desde múltiples escalas territoriales**, como:
- **Estado o departamento**
- **Ciudad o municipio**
- **Localidad, barrio o vereda**

Esto habilita casos de uso como:
- Mapas interactivos agregados por zona.
- Tableros de control con filtros por región.
- Análisis de distribución espacial de eventos viales o condiciones anómalas.

---

### ✅ Funcionalidades principales:

- 📂 **Lectura automatizada**: recorre la carpeta donde se encuentran los archivos `.txt` por segmento (generados previamente).
- 🧩 **Unificación de archivos**: concatena todos los archivos en un único `DataFrame` con trazabilidad por archivo (`Archivo`).
- 💾 **Exportación tabular**:
  - `locations2.csv` para análisis con Python o Power BI.
  - `locations2.xlsx` para revisión en Excel u hojas de cálculo compartidas.

---

### 🧠 Casos de uso:

- Visualizar todos los segmentos de **vía** sobre un mapa clasificado por ciudad.
- Agrupar la cantidad de **automóviles detectados** por barrio o zona urbana.
- Mapear las **anomalías** (baches, obstáculos, sombreado) por departamento o localidad.
- Integrar los datos en sistemas SIG o plataformas de análisis geográfico.

---

### 🛠️ Resultado final:

Un archivo único con columnas estandarizadas por segmento, incluyendo:
- Coordenadas (`lat`, `lon`)
- Ubicación detallada (`road`, `suburb`, `municipality`, `state`, `country`, etc.)
- Identificador de archivo de origen para trazabilidad

Este archivo puede ser utilizado directamente en Power BI, Tableau, QGIS o Dashboards personalizados para **visualización espacial avanzada** de todos los resultados obtenidos en la caracterización vial.


In [1]:
import os,glob
from tqdm import tqdm
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon
from xml.etree import ElementTree as ET
import math
from shapely.ops import transform
import pyproj
import leafmap

## PARÁMETROS
metros=100
extension_in_meters = 22#15  # Define la distancia en metros


root_folder=r'C:\Users\Sebastian\Documents\CORREDORES_V'
ruta_kml = os.path.join(root_folder,'Troncales_V.kml')  # Reemplaza con la ruta de tu archivo
meters_to_degrees = extension_in_meters / 111320  # Conversión aproximada de metros a grados (latitud)

## FUNCIONES
def calcular_bounding_box(linestring, extension):
    if linestring.is_empty:
        return None

    # Obtener los límites del LineString
    minx, miny, maxx, maxy = linestring.bounds

    # Extender los límites
    minx -= extension
    miny -= extension
    maxx += extension
    maxy += extension

    # Crear el bounding box como lista
    return [minx, miny, maxx, maxy]

def calcular_longitud(line):
    coords = list(line.coords)
    total_length_m = 0.0
    for i in range(len(coords)-1):
        lon1, lat1 = coords[i][0], coords[i][1]
        lon2, lat2 = coords[i+1][0], coords[i+1][1]
        _, _, dist = geod.inv(lon1, lat1, lon2, lat2)
        total_length_m += dist
    return total_length_m

# Función para eliminar la Z de las geometrías (opcional si existen Z)
def drop_z(geom):
    if geom.has_z:
        return LineString([(x, y) for (x, y, z) in geom.coords])
    else:
        return geom

# Parsear el archivo KML
tree = ET.parse(ruta_kml)
root = tree.getroot()

# Definir el namespace del KML
namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}

# Extraer todos los placemarks
placemarks = root.findall(".//kml:Placemark", namespaces)

# Inicializar listas para almacenar datos
names = []
geometries = []

# Procesar cada placemark
for placemark in placemarks:
    # Obtener el nombre
    name = placemark.find("kml:name", namespaces)
    name = name.text if name is not None else "Sin nombre"

    # Obtener las coordenadas
    coordinates = placemark.find(".//kml:coordinates", namespaces)
    if coordinates is not None:
        coord_text = coordinates.text.strip()
        coords = [
            tuple(map(float, coord.split(',')))
            for coord in coord_text.split()
        ]

        # Determinar el tipo de geometría según la cantidad de puntos
        if len(coords) == 1:
            geometries.append(Point(coords[0]))
        else:
            geometries.append(LineString(coords))
    else:
        geometries.append(None)

    # Agregar el nombre
    names.append(name)

# Crear un GeoDataFrame
gdf = gpd.GeoDataFrame({'Name': names, 'geometry': geometries}, crs="EPSG:4326")
#gdf.to_file("salida.geojson", driver="GeoJSON")

# Filtrar por geometrías de tipo Polygon
gdf_rutas = gdf[gdf.geometry.apply(lambda geom: isinstance(geom, LineString))]

# Opcional: Guardar el GeoDataFrame filtrado
out_geojson=ruta_kml.replace('.kml','.geojson')
out_shp=ruta_kml.replace('.kml','.shp')
#gdf_rutas.to_file(out_geojson, driver="GeoJSON")
#gdf_rutas.to_file(out_shp)

gdf_rutas = gdf_rutas.to_crs(epsg=4326)
geod = pyproj.Geod(ellps='WGS84')

# Aplica la función a cada geometría del gdf
gdf_rutas['length'] = gdf_rutas.geometry.apply(calcular_longitud)


# Proyectar a EPSG:3857 para medir en metros
gdf_rutas_3857 = gdf_rutas.copy()
gdf_rutas_3857['geometry'] = gdf_rutas_3857['geometry'].apply(drop_z)
gdf_rutas_3857 = gdf_rutas_3857.to_crs(epsg=3857)

segmentos = []

# Recorremos cada fila (cada línea)
for idx, row in gdf_rutas_3857.iterrows():
    line = row.geometry
    length_m = line.length
    name = row['Name']  # Conservar la columna Name
    # Numero de segmentos = entero superior de la longitud/1000
    n_segments = math.ceil(length_m / metros)

    # Distancias a lo largo de la línea donde obtendremos puntos (cada 1000 m)
    distances = [i*metros for i in range(n_segments)]
    if distances[-1] < length_m:
        distances.append(length_m)

    # Interpolamos puntos a cada distancia
    points = [line.interpolate(d) for d in distances]

    # Creamos segmentos de ~1 km
    for i in range(len(points)-1):
        seg = LineString([points[i], points[i+1]])
        seg_length_km = seg.length  # Convertir la longitud del segmento a km
        # Conservar original_idx, Name y asignar length_km
        segmentos.append((idx, seg, seg_length_km, name))

# Creamos un nuevo gdf con los segmentos, su longitud y el nombre original
gdf_segmentos = gpd.GeoDataFrame(segmentos, columns=['original_idx', 'geometry', 'length', 'Name'], crs=gdf_rutas_3857.crs)

# Reproyectamos de vuelta a EPSG:4326
gdf_segmentos = gdf_segmentos.to_crs(epsg=4326)

# Crear la columna 'bb' en el GeoDataFrame
gdf_segmentos['bb'] = gdf_segmentos.geometry.apply(lambda geom: calcular_bounding_box(geom, meters_to_degrees))

# Ahora gdf_segmentos tiene aproximadamente 1km por segmento, una columna length_km con su distancia, y conserva la columna Name
print(gdf_segmentos.shape,gdf_segmentos.columns)
print(gdf_segmentos['original_idx'].value_counts())
print(gdf_segmentos['Name'].value_counts())
display(gdf_segmentos.head())

#Exportar un csv
out_csv=ruta_kml.replace('.kml','.csv')
#gdf_segmentos['geometry'] = gdf_segmentos['geometry'].apply(lambda x: x.wkt)
#gdf_segmentos.to_csv(out_csv)

m = leafmap.Map()
m.add_gdf(gdf_segmentos, layer_name="Capa KML")
m

(37881, 5) Index(['original_idx', 'geometry', 'length', 'Name', 'bb'], dtype='object')
original_idx
3     9219
19    6684
0     6217
16    5911
7     5292
11    4558
Name: count, dtype: int64
Name
Troncal1_SanCristobal_Caracas                    9219
Troncal9_Caracas_Guiria                          6684
Troncal5_CampoCarabobo,_SanCristobal             6217
Troncal2_Puerto Ayacucho_SanJuanMorros           5911
Troncal3_PuertoCabello_Coro_Maracaibo            5292
Troncal4_Guanare_Barquisimeto_Churuguara_Coro    4558
Name: count, dtype: int64


,original_idx,geometry,length,Name,bb
0,0,"LINESTRING (-68.15619 10.01298, -68.15669 10.0...",84.236227,"Troncal5_CampoCarabobo,_SanCristobal","[-68.15688658328916, 10.012222134417827, -68.1..."
1,0,"LINESTRING (-68.15669 10.01242, -68.15742 10.0...",100.000000,"Troncal5_CampoCarabobo,_SanCristobal","[-68.15761997954047, 10.011711285741615, -68.1..."
2,0,"LINESTRING (-68.15742 10.01191, -68.15812 10.0...",97.175338,"Troncal5_CampoCarabobo,_SanCristobal","[-68.15831502366083, 10.01119118090159, -68.15..."
3,0,"LINESTRING (-68.15812 10.01139, -68.15894 10.0...",100.000000,"Troncal5_CampoCarabobo,_SanCristobal","[-68.1591408147139, 10.010842955817411, -68.15..."
4,0,"LINESTRING (-68.15894 10.01104, -68.15977 10.0...",99.994634,"Troncal5_CampoCarabobo,_SanCristobal","[-68.15996915519679, 10.010500775564672, -68.1..."


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [2]:
from geopy.geocoders import Nominatim

output_folder = r"C:\Users\Sebastian\Documents\CORREDORES_V\Registros_Locaciones2"
os.makedirs(output_folder, exist_ok=True)

# Inicializar geolocalizador
geolocalizador = Nominatim(user_agent="BEFM")

In [ ]:
import os
import ast
from tqdm import tqdm


# Iterar sobre los datos
for idx, row in tqdm(gdf_segmentos.iterrows(), total=len(gdf_segmentos)):
    # Extraer bounding box y calcular centroide
    bb = ast.literal_eval(row['bb']) if isinstance(row['bb'], str) else row['bb']
    centroid = ((bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2)
    lon, lat = centroid[0], centroid[1]

    # Definir el nombre del archivo con el Name del segmento
    file_name = f"{row['Name'].split('_')[0]}_{idx}.txt"
    print(file_name)
    file_path = os.path.join(output_folder, file_name)

    # Verificar si el archivo ya existe
    if os.path.exists(file_path):
        print(f"El archivo {file_name} ya existe. Saltando...")
        continue  # Omitir si el archivo ya existe

    try:
        # Obtener ubicación
        ubicacion = geolocalizador.reverse((lat, lon), exactly_one=True)
        direccion = ubicacion.raw.get('address', {})

        # Extraer información requerida
        datos = [
            row['Name'], str(lat), str(lon),
            direccion.get('road', ''),
            direccion.get('suburb', ''),
            direccion.get('village', ''),
            direccion.get('municipality', ''),
            direccion.get('county', ''),
            direccion.get('state', ''),
            direccion.get('ISO3166-2-lvl4', ''),
            direccion.get('postcode', ''),
            direccion.get('country', ''),
            direccion.get('country_code', '')
        ]

        # Escribir datos en el archivo de texto
        with open(file_path, "w", encoding="utf-8") as f:
            columnas = ['Name', 'lat', 'lon', 'road', 'suburb', 'village', 'municipality', 
                        'county', 'state', 'ISO3166-2-lvl4', 'postcode', 'country', 'country_code']
            f.write("\t".join(columnas) + "\n")  # Escribir encabezado
            f.write("\t".join(datos) + "\n")  # Escribir datos

    except Exception as e:
        print(f"Error al procesar {row['Name']} ({lat}, {lon}): {e}")

# UNIR TXT FILES

In [7]:
import os
import pandas as pd

# Ruta de la carpeta donde están los archivos de texto
folder_path = output_folder
# Lista para acumular los datos de cada archivo
datos = []

# Recorre todos los archivos en la carpeta
for file_name in os.listdir(folder_path):
    if file_name.endswith('.txt'):
        file_path = os.path.join(folder_path, file_name)
        
        # Leer el archivo como DataFrame
        df_temp = pd.read_csv(file_path, delimiter='\t')  # Archivos separados por tabulaciones
        
        # Agregar columna con el nombre del archivo
        df_temp["Archivo"] = file_name
        
        # Acumular datos
        datos.append(df_temp)

# Concatenar todos los DataFrames en uno solo
df_final = pd.concat(datos, ignore_index=True)

# Guardar los datos en CSV y Excel
out_csv = os.path.join(folder_path, "locations2.csv")
out_excel = os.path.join(folder_path, "locations2.xlsx")
df_final.to_csv(out_csv, index=False)
df_final.to_excel(out_excel, index=False)

print("Proceso completado. Archivos guardados:")
print(out_csv)
print(out_excel)


Proceso completado. Archivos guardados:
C:\Users\Sebastian\Documents\CORREDORES_V\Registros_Locaciones2\locations2.csv
C:\Users\Sebastian\Documents\CORREDORES_V\Registros_Locaciones2\locations2.xlsx
